In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import re  # 用于正则表达式

def get_file_order(input_dir):
    """返回按照月份和编号排序后的文件列表"""
    files = [f for f in os.listdir(input_dir) if f.endswith('.xlsx') and f.startswith('data') and '_charge' in f]
    
    # 提取文件中的月份和编号进行排序，确保处理正确的格式
    def extract_month_and_number(file_name):
        # 使用正则表达式提取月份和编号
        match = re.match(r'data(\d+)_charge(\d+)', file_name)
        if match:
            month = int(match.group(1))  # 提取月份
            number = int(match.group(2))  # 提取编号
            return (month, number)
        return None  # 如果文件名不符合预期格式，返回 None

    # 过滤不符合预期格式的文件
    valid_files = [f for f in files if extract_month_and_number(f) is not None]
    
    # 使用提取的月份和编号进行排序
    files_sorted = sorted(valid_files, key=lambda f: extract_month_and_number(f))
    
    return files_sorted

def process_files_in_folder():
    folder_path = os.path.join(os.getcwd(), 'charge')  # 获取当前工作目录下的'charge'文件夹路径
    files = get_file_order(folder_path)  # 获取按顺序排列的文件列表

    deep_char_accum = 0
    fastchgnum_accum = 0

    prev_deep_char_val = 0
    prev_fastchgnum_val = 0

    # 使用 tqdm 添加进度条
    for file in tqdm(files, desc="Processing Files", unit="file"):
        file_path = os.path.join(folder_path, file)
        
        # 检查文件是否存在
        if not os.path.exists(file_path):
            print(f"File not found: {file}")
            continue

        # 读取Excel文件
        df = pd.read_excel(file_path, engine='openpyxl')

        # 初始化新列
        deep_char_col = [prev_deep_char_val] * len(df)
        fastchgnum_col = [prev_fastchgnum_val] * len(df)
        
        deep_char_accum = 0
        fastchgnum_accum = 0

        if len(df) > 5:  # 确保有足够的行数
            second_col = df.iloc[1:6, 6].astype(float)
            last_five_rows = df.iloc[-5:, 6].astype(float)

            if second_col.mean() < 15:
                deep_char_accum += 1
            if last_five_rows.mean() > 94:
                deep_char_accum += 1

            deep_char_col = [prev_deep_char_val + deep_char_accum] * len(df)

        if df.shape[1] > 23:  # 确保有足够的列数
            if (df.iloc[:, 23].astype(float) >= 40).any():
                fastchgnum_accum += 1

            fastchgnum_col = [prev_fastchgnum_val + fastchgnum_accum] * len(df)

        # 将新列添加到数据框
        df['deep_char'] = deep_char_col
        df['FastChgNum'] = fastchgnum_col

        # 保存更新后的Excel文件
        df.to_excel(file_path, index=False, engine='openpyxl')

        # 更新前一文件的列值
        prev_deep_char_val = deep_char_col[0]
        prev_fastchgnum_val = fastchgnum_col[0]

# 调用函数
process_files_in_folder()
